In [1]:
# --- BLOQUE INICIAL ---
import os
import xarray as xr
import geopandas as gpd
import rioxarray  # si queremos recortar usando geometrías shapely
import pickle
import pandas as pd
import numpy as np


In [2]:
BASE_DIR = os.path.abspath("..")
INPUT_DIR = os.path.join(BASE_DIR, "data", "processed", "pH")
OUTPUT_DIR = INPUT_DIR  # guardamos en el mismo lugar


In [5]:
# Shapes
SHAPES_DIR = os.path.join(BASE_DIR, "data", "raw", "shapes")
bbi_shape = gpd.read_file(os.path.join(SHAPES_DIR, "transicion_solo.shp"))
bbii_shape = gpd.read_file(os.path.join(SHAPES_DIR, "BBII_unificado.shp"))

## PARA PH

In [6]:
# --- Parámetros del filtro físico ---
# pH superficial oceánico: rango físicamente posible en el Atlántico Sur subantártico
# Valores fuera de este rango son artefactos numéricos del modelo
PH_MIN = 7.5
PH_MAX = 8.5

In [10]:
# =============================================================================
# FUNCIÓN DE PREPROCESAMIENTO
# =============================================================================
 
def preprocess_ph(ds):
    """
    Preprocesamiento mínimo para datos de pH de CMEMS MULTIOBS.
 
    Pasos:
    1. Ordenar por tiempo
    2. Eliminar timestamps duplicados
    3. Filtro físico conservador: pH fuera de (PH_MIN, PH_MAX) → NaN
 
    No se interpolan NaN ni se aplican filtros estadísticos adicionales.
    Los NaN son manejados nativamente por xarray (skipna=True en promedios).
    """
    # 1. Ordenar tiempo
    ds = ds.sortby("time")
 
    # 2. Eliminar duplicados temporales
    ds = ds.drop_duplicates(dim="time")
 
    # 3. Filtro físico: solo sobre la variable pH
    #    El resto de variables (spco2, talk, tco2, omega) NO se filtran aquí
    #    porque tienen sus propios rangos físicos — se filtrarán si se usan
    ds["ph"] = ds["ph"].where(
        (ds["ph"] > PH_MIN) & (ds["ph"] < PH_MAX)
    )
 
    return ds

In [12]:
# =============================================================================
# APLICAR A LOS TRES ARCHIVOS
# =============================================================================
 
files = {
    "ph_BBI.nc":   "ph_BBI_ready.nc",    # series temporales — BBI (plateau)
    "ph_BBII.nc":  "ph_BBII_ready.nc",   # series temporales — BBII (slope)
    "ph_bbox.nc":  "ph_bbox_ready.nc",   # mapas de estrés — grilla completa
}
 
for input_file, output_file in files.items():
    input_path  = os.path.join(INPUT_DIR, input_file)
    output_path = os.path.join(OUTPUT_DIR, output_file)
 
    print(f"\n{'='*60}")
    print(f"Procesando: {input_file}")
 
    ds = xr.open_dataset(input_path)
 
    # Verificar NaN antes
    n_nan_before = int(ds["ph"].isnull().sum())
 
    ds_clean = preprocess_ph(ds)
 
    # Verificar NaN después
    n_nan_after = int(ds_clean["ph"].isnull().sum())
    n_filtered  = n_nan_after - n_nan_before
 
    print(f"  Dimensiones  : {dict(ds_clean.dims)}")
    print(f"  Período      : {str(ds_clean.time.values[0])[:10]} → "
          f"{str(ds_clean.time.values[-1])[:10]}")
    print(f"  NaN antes    : {n_nan_before}")
    print(f"  NaN después  : {n_nan_after}  (+{n_filtered} filtrados por rango físico)")
    print(f"  pH min/max   : {float(ds_clean['ph'].min()):.4f} / "
          f"{float(ds_clean['ph'].max()):.4f}")
 
    ds_clean.to_netcdf(output_path)
    print(f"  ✅ Guardado  : {output_file}")
 
print(f"\n{'='*60}")
print("Preprocesamiento completado.")
print(f"Archivos generados en: {OUTPUT_DIR}")
print(f"\nCriterio de filtrado documentado para el paper:")
print(f"  'Values outside the physically plausible range "
      f"({PH_MIN} < pH < {PH_MAX}) were excluded.'")


Procesando: ph_BBI.nc


C:\Users\gisel\AppData\Local\Temp\ipykernel_216748\1685443547.py:29: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensiones  : {dict(ds_clean.dims)}")
C:\Users\gisel\AppData\Local\Temp\ipykernel_216748\1685443547.py:29: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensiones  : {dict(ds_clean.dims)}")


  Dimensiones  : {'time': 480, 'latitude': 14, 'longitude': 37}
  Período      : 1985-01-01 → 2024-12-01
  NaN antes    : 217440
  NaN después  : 217440  (+0 filtrados por rango físico)
  pH min/max   : 8.0158 / 8.1649
  ✅ Guardado  : ph_BBI_ready.nc

Procesando: ph_BBII.nc
  Dimensiones  : {'time': 480, 'latitude': 14, 'longitude': 37}
  Período      : 1985-01-01 → 2024-12-01
  NaN antes    : 220800
  NaN después  : 220800  (+0 filtrados por rango físico)
  pH min/max   : 8.0228 / 8.1576
  ✅ Guardado  : ph_BBII_ready.nc

Procesando: ph_bbox.nc
  Dimensiones  : {'time': 480, 'latitude': 14, 'longitude': 37}
  Período      : 1985-01-01 → 2024-12-01
  NaN antes    : 0
  NaN después  : 0  (+0 filtrados por rango físico)
  pH min/max   : 8.0152 / 8.2006
  ✅ Guardado  : ph_bbox_ready.nc

Preprocesamiento completado.
Archivos generados en: C:\Users\gisel\BB_stress_paper\data\processed\pH

Criterio de filtrado documentado para el paper:
  'Values outside the physically plausible range (7.5 < 

C:\Users\gisel\AppData\Local\Temp\ipykernel_216748\1685443547.py:29: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensiones  : {dict(ds_clean.dims)}")


In [13]:
print(ds.data_vars)

Data variables:
    fgco2                 (time, latitude, longitude) float32 995kB ...
    fgco2_uncertainty     (time, latitude, longitude) float32 995kB ...
    omega_ar              (time, latitude, longitude) float32 995kB ...
    omega_ar_uncertainty  (time, latitude, longitude) float32 995kB ...
    omega_ca              (time, latitude, longitude) float32 995kB ...
    omega_ca_uncertainty  (time, latitude, longitude) float32 995kB ...
    ph                    (time, latitude, longitude) float32 995kB 8.136 ......
    ph_uncertainty        (time, latitude, longitude) float32 995kB ...
    spco2                 (time, latitude, longitude) float32 995kB ...
    spco2_uncertainty     (time, latitude, longitude) float32 995kB ...
    talk                  (time, latitude, longitude) float32 995kB ...
    talk_uncertainty      (time, latitude, longitude) float32 995kB ...
    tco2                  (time, latitude, longitude) float32 995kB ...
    tco2_uncertainty      (time, latitu

In [14]:
# --- Rangos físicos por variable ---
# Basados en la distribución real de tus datos (celda 3 de 02b)
# y rangos típicos del Atlántico Sur subantártico
RANGES = {
    "ph"       : (7.5,   8.5),
    "omega_ar" : (0.5,   3.5),
    "omega_ca" : (0.8,   5.5),
    "talk"     : (2100., 2400.),
    "tco2"     : (1900., 2250.),
    "spco2"    : (100.,  700.),
    "fgco2"    : (-10.,  10.),   # mol/m²/año, puede ser negativo (sumidero)
}
# Las variables de incertidumbre (_uncertainty) no se filtran

def preprocess_dataset(ds):
    """
    Preprocesamiento mínimo para todas las variables del producto
    CMEMS MULTIOBS.

    Pasos:
    1. Ordenar por tiempo
    2. Eliminar timestamps duplicados
    3. Filtro físico conservador por variable → NaN fuera del rango

    Las variables de incertidumbre (_uncertainty) no se filtran —
    su rango depende del modelo y no tiene interpretación física directa.
    """
    # 1. Ordenar tiempo
    ds = ds.sortby("time")

    # 2. Eliminar duplicados temporales
    ds = ds.drop_duplicates(dim="time")

    # 3. Filtro físico por variable
    for var, (vmin, vmax) in RANGES.items():
        if var in ds:
            n_before = int(ds[var].isnull().sum())
            ds[var]  = ds[var].where((ds[var] > vmin) & (ds[var] < vmax))
            n_after  = int(ds[var].isnull().sum())
            if n_after > n_before:
                print(f"    {var}: {n_after - n_before} valores filtrados")

    return ds


# --- Aplicar a los tres archivos ---
files = {
    "ph_BBI.nc"  : "ph_BBI_ready.nc",
    "ph_BBII.nc" : "ph_BBII_ready.nc",
    "ph_bbox.nc" : "ph_bbox_ready.nc",
}

for input_file, output_file in files.items():
    input_path  = os.path.join(INPUT_DIR, input_file)
    output_path = os.path.join(OUTPUT_DIR, output_file)

    print(f"\n{'='*60}")
    print(f"Procesando: {input_file}")

    ds       = xr.open_dataset(input_path)
    ds_clean = preprocess_dataset(ds)

    print(f"  Período : {str(ds_clean.time.values[0])[:10]} → "
          f"{str(ds_clean.time.values[-1])[:10]}")
    print(f"  ✅ Guardado: {output_file}")

    ds_clean.to_netcdf(output_path)

print(f"\n{'='*60}")
print("Preprocesamiento completado.")
print("\nCriterios de filtrado documentados para el paper:")
for var, (vmin, vmax) in RANGES.items():
    print(f"  {var}: ({vmin}, {vmax})")


Procesando: ph_BBI.nc
  Período : 1985-01-01 → 2024-12-01
  ✅ Guardado: ph_BBI_ready.nc

Procesando: ph_BBII.nc
  Período : 1985-01-01 → 2024-12-01
  ✅ Guardado: ph_BBII_ready.nc

Procesando: ph_bbox.nc
  Período : 1985-01-01 → 2024-12-01
  ✅ Guardado: ph_bbox_ready.nc

Preprocesamiento completado.

Criterios de filtrado documentados para el paper:
  ph: (7.5, 8.5)
  omega_ar: (0.5, 3.5)
  omega_ca: (0.8, 5.5)
  talk: (2100.0, 2400.0)
  tco2: (1900.0, 2250.0)
  spco2: (100.0, 700.0)
  fgco2: (-10.0, 10.0)
